#  Code to import the data_pipeline and visualize data sets

In [1]:
import sys
import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go


sys.path.append(r"C:\Users\arbaz2\Desktop\Quant Finance\Volatility Forecasting\src")

from data_pipeline import make_dataset

In [2]:
CRISIS_WINDOWS = {
    "GFC_2007_2009": ("2007-07-01", "2009-06-30"),
    "COVID_2020": ("2020-02-15", "2020-05-31"),
}

df = make_dataset(["SPY","JPM"], start="2000-01-01", crisis_windows=CRISIS_WINDOWS)
df["date"] = pd.to_datetime(df["date"])
df.head()

,date,ticker,open,high,low,close,adj_close,volume,ret,ret2,rv1_var,rv5_var,regime
0,2000-01-10,JPM,48.500000,48.916668,47.666668,47.666668,22.389442,4723500,-1.733139,3.003770,5.704408,14.393516,calm
1,2000-01-10,SPY,146.250000,146.906250,145.031250,146.250000,91.641884,5741700,0.342478,0.117291,1.449138,36.092597,calm
2,2000-01-11,JPM,46.666668,46.958332,45.500000,46.541668,21.861031,8405550,-2.388390,5.704408,0.390121,14.400174,calm
3,2000-01-11,SPY,145.812500,146.093750,143.500000,144.500000,90.545311,7503700,-1.203802,1.449138,0.999640,37.060292,calm
4,2000-01-12,JPM,46.458332,47.250000,46.333332,46.833332,21.998001,7271850,0.624596,0.390121,2.253657,14.666753,calm


In [3]:
# Check if GFC and COVID rows exist and majority are calm
df["regime"].value_counts()

regime
calm             12236
GFC_2007_2009     1008
COVID_2020         144
Name: count, dtype: int64

#### Helper functions to add a shaded region for crisis period and to filter data for one ticker to keeps things tidy

In [ ]:
def add_crisis_shading(fig, crisis_windows=CRISIS_WINDOWS, opacity=0.45):
    for name, (start, end) in crisis_windows.items():
        fig.add_vrect(
            x0=pd.to_datetime(start),
            x1=pd.to_datetime(end),
            fillcolor="pink",
            opacity=opacity,
            line_width=0,
            annotation_text=name,
            annotation_position="top left"
        )
    return fig

def one_ticker(df, ticker):
    return df[df["ticker"] == ticker].sort_values("date").copy()

In [14]:
def plot_price(df, ticker):
    d = one_ticker(df, ticker)
    fig = px.line(d, x="date", y="adj_close", title=f"{ticker} Adjusted Close")
    fig.update_layout(xaxis_title="Date", yaxis_title="Adj Close")
    add_crisis_shading(fig)
    fig.show()

plot_price(df, "SPY")
plot_price(df, "JPM")

In [16]:
def plot_returns(df, ticker):
    d = one_ticker(df, ticker)
    fig = px.line(d, x="date", y="ret", title=f"{ticker} Daily Log Returns (%)")
    fig.update_layout(xaxis_title="Date", yaxis_title="Return (%)")
    add_crisis_shading(fig)
    fig.show()

plot_returns(df, "SPY")
plot_returns(df, "JPM")

In [17]:
def plot_return_hist(df, ticker, nbins=140):
    d = df[df["ticker"] == ticker]
    fig = px.histogram(d, x="ret", nbins=nbins, title=f"{ticker} Return Histogram")
    fig.update_layout(xaxis_title="Return (%)", yaxis_title="Count")
    fig.show()

plot_return_hist(df, "SPY")
plot_return_hist(df, "JPM")

In [18]:
def plot_squared_returns(df, ticker):
    d = one_ticker(df, ticker)
    d["ret2"] = d["ret"]**2
    fig = px.line(d, x="date", y="ret2", title=f"{ticker} Squared Returns (ret²)")
    fig.update_layout(xaxis_title="Date", yaxis_title="ret²")
    add_crisis_shading(fig)
    fig.show()

plot_squared_returns(df, "SPY")

In [15]:
def plot_targets(df, ticker):
    d = one_ticker(df, ticker)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=d["date"], y=d["rv1_var"], mode="lines", name="rv1_var (1d)"))
    fig.add_trace(go.Scatter(x=d["date"], y=d["rv5_var"], mode="lines", name="rv5_var (5d)"))
    fig.update_layout(
        title=f"{ticker} Realized Variance Targets",
        xaxis_title="Date",
        yaxis_title="Variance proxy"
    )
    add_crisis_shading(fig)
    fig.show()

plot_targets(df, "SPY")

In [12]:
def plot_rolling_vol(df, ticker, window=20):
    d = one_ticker(df, ticker)
    d["roll_var"] = d["ret"].rolling(window).var()
    d["roll_vol"] = np.sqrt(d["roll_var"])
    fig = px.line(d, x="date", y="roll_vol", title=f"{ticker} {window}D Rolling Vol (from returns)")
    fig.update_layout(xaxis_title="Date", yaxis_title="Vol (return units)")
    add_crisis_shading(fig)
    fig.show()

plot_rolling_vol(df, "SPY", window=20)
plot_rolling_vol(df, "JPM", window=20)

In [17]:
def plot_rolling_vol_multi(df, window=20):
    d = df.sort_values(["ticker", "date"]).copy()
    d["roll_var"] = d.groupby("ticker")["ret"].transform(lambda s: s.rolling(window).var())
    d["roll_vol"] = np.sqrt(d["roll_var"])
    d = d.dropna(subset=["roll_vol"]).sort_values(["date", "ticker"])

    fig = px.line(d, x="date", y="roll_vol", color="ticker", title=f"{window}D Rolling Vol Comparison")
    fig.update_layout(xaxis_title="Date", yaxis_title="Vol (return units)")
    add_crisis_shading(fig)
    fig.show()

plot_rolling_vol_multi(df, window=20)

In [23]:
def plot_zoom(df, ticker, start, end, y="roll_vol", window=20, title=None):
    d = one_ticker(df, ticker)
    d["roll_var"] = d["ret"].rolling(window).var()
    d["roll_vol"] = np.sqrt(d["roll_var"])
    d = d[(d["date"] >= pd.to_datetime(start)) & (d["date"] <= pd.to_datetime(end))]

    fig = px.line(d, x="date", y=y, title=title or f"{ticker} {y} ({start} to {end})")
    fig.update_layout(xaxis_title="Date", yaxis_title=y)
    fig.show()

# GFC zoom
plot_zoom(df, "SPY", "2007-07-01", "2009-06-30", y="roll_vol", window=20, title="SPY 20D Rolling Vol — GFC")
# COVID zoom
plot_zoom(df, "SPY", "2020-02-15", "2020-05-31", y="roll_vol", window=20, title="SPY 20D Rolling Vol — COVID")